# 🏥 Notebook 24: Interactive Hospital Portal & Clinical Visual Analytics
## Enterprise Clinical Operations Dashboard, Interactive Visualizations & Live SQL Querying

**Author:** CareFlow Clinical Intelligence & Data Visualization Group  
**Pillars:** Data Visualization, Clinical Operations, Relational SQL & Healthcare Informatics  

---

### Overview
This notebook represents the pure Python foundation of the **Hospital Clinical Operations Portal**. It unifies:
1. **Live SQL Database Ingestion:** Querying relational tables (`fct_patient_journeys`, `stg_ehr_events`, `hospital_ward_census`).
2. **Interactive Clinical Visual Analytics:** Plotly horizontal bed saturation gauges, triage wait boxplots, and care flow funnels.
3. **Real-Time Patient Intake:** Transactional SQL operations adding new clinical episodes.
4. **Clinical Decision Support:** Automated patient discharge instructions and SBAR handovers.

In [ ]:
# 1. Environment Setup & Core Libraries
import os
import sys
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

sys.path.insert(0, os.path.abspath('..'))
from careflow_analytics.database import query_sql, get_database_stats

print("[OK] Hospital Portal Analytics Engine initialized.")

### 2. Live Inpatient Ward Census Query
Querying active bed occupancy across intensive care, step-down, and surgical units.

In [ ]:
census_df = query_sql("SELECT * FROM hospital_ward_census ORDER BY occupancy_rate DESC")
census_df[['unit_name', 'total_beds', 'occupied_beds', 'occupancy_rate', 'status']]

### 3. Ward Capacity Visualization with Surge Thresholds
Generating institutional bed capacity telemetry with 85% Surge and 90% Critical saturation lines.

In [ ]:
fig_census = go.Figure()
bar_colors = [
    '#ef4444' if r >= 90.0 else ('#f59e0b' if r >= 80.0 else '#10b981')
    for r in census_df['occupancy_rate']
]
fig_census.add_trace(go.Bar(
    y=census_df['unit_name'],
    x=census_df['occupancy_rate'],
    orientation='h',
    marker=dict(color=bar_colors),
    text=[f"{r:.1f}% ({o}/{t} beds)" for r, o, t in zip(census_df['occupancy_rate'], census_df['occupied_beds'], census_df['total_beds'])],
    textposition='outside'
))
fig_census.add_vline(x=85.0, line_dash="dash", line_color="#f59e0b", annotation_text="Surge Warning (85%)")
fig_census.add_vline(x=90.0, line_dash="dash", line_color="#ef4444", annotation_text="Critical Saturation (90%)")
fig_census.update_layout(
    title="Hospital Inpatient Ward Saturation Telemetry",
    xaxis_title="Bed Occupancy Rate (%)",
    yaxis=dict(autorange="reversed"),
    template="plotly_dark",
    height=400
)
fig_census.show()

### 4. Triage Door-to-Doctor Regulatory Compliance (CMS OP-18)
Benchmarking doctor consultation wait-times against the mandatory 30-minute clinical threshold.

In [ ]:
sla_df = query_sql("SELECT * FROM mart_sla_compliance")

fig_box = px.box(
    sla_df,
    x="clinical_severity",
    y="door_to_doctor_minutes",
    color="clinical_severity",
    title="Door-to-Doctor Wait Times by Triage Acuity (30m Target)",
    template="plotly_dark",
    color_discrete_sequence=['#f43f5e', '#f59e0b', '#10b981']
)
fig_box.add_hline(y=30.0, line_dash="dash", line_color="#ef4444", annotation_text="Regulatory Mandate (30m)")
fig_box.update_layout(height=380)
fig_box.show()

### 5. Patient Flow Throughput Funnel
Stage-by-stage patient volume progression from emergency room check-in to inpatient disposition.

In [ ]:
stages_sql = """
SELECT activity_name AS stage, COUNT(DISTINCT case_id) AS patient_volume
FROM stg_ehr_events
WHERE activity_name IN ('Registration', 'Triage', 'Doctor Consultation', 'X-Ray', 'Treatment')
GROUP BY activity_name
ORDER BY patient_volume DESC;
"""
funnel_df = query_sql(stages_sql)

fig_funnel = px.funnel(
    funnel_df,
    x='patient_volume',
    y='stage',
    title="Emergency Department Care Stage Retention Funnel",
    template="plotly_dark",
    color_discrete_sequence=['#06b6d4']
)
fig_funnel.update_layout(height=380)
fig_funnel.show()

### Conclusion & Portal Implementation
- The hospital portal combines relational SQL data persistence with interactive clinical visualizations.
- Clinicians, quality officers, and hospital leadership can monitor bed shortages, SLA compliance, and throughput in real time.